# AWS Credit card fraud detection 

In this solution we will build the core of a credit card fraud detection system using SageMaker. We will start by training an anomaly detection algorithm, then proceed to train two XGBoost models for supervised training. To deal with the highly unbalanced data common in fraud detection, our first model will use re-weighting of the data, and the second will use re-sampling, using the popular SMOTE technique for oversampling the rare fraud data.

Our solution includes an example of making calls to a REST API to simulate a real deployment, using AWS Lambda to trigger both the anomaly detection and XGBoost model.

You can select Run->Run All from the menu to run all cells in Studio (or Cell->Run All in a SageMaker Notebook Instance).

**Note**: When running this notebook on SageMaker Studio, you should make sure the 'SageMaker JumpStart Data Science 1.0' image/kernel is used.

In [ ]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import IsolationForest
import xgboost as xgb
import boto3
import joblib
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import sys
sys.path.insert(0, '.')

### Set up environment

Let's set up environment

In [ ]:
# Configuration des variables d'environnement
aws_region = os.environ.get('AWS_REGION')
aws_access_key = os.getenv("AWS_ID_ACCESS_KEY")
aws_secret_key = os.getenv("AWS_SECRET_ACCESS_KEY")
s3_bucket = os.getenv("SOLUTIONS_S3_BUCKET")
s3_prefix = os.getenv("SOLUTION_NAME")

print(f"aws_region: {aws_region}")
print(f"aws_access_key: {aws_access_key}")
print(f"aws_secret_key: {aws_secret_key}")
print(f"s3_bucket: {s3_bucket}")
print(f"s3_prefix: {s3_prefix}")

In [ ]:
DATASET_PATH = 'dataset'
os.makedirs(DATASET_PATH, exist_ok=True)
# os.makedirs(CHECKPOINTS_PATH, exist_ok=True)

In [ ]:
# Initialisation du client S3
s3_client = boto3.client(
    's3',
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_key,
    region_name=aws_region
)

In [ ]:
# Download file from S3
s3_key = f"{s3_prefix}/dataset/creditcard.csv.zip"
local_zip_path = f"{DATASET_PATH}/creditcard.csv.zip"

print("Téléchargement en cours...")
s3_client.download_file(s3_bucket, s3_key, local_zip_path)
print(f"Téléchargement terminé : {local_zip_path}")

In [ ]:
# Unzip file to DATASET_PATH
print("Décompression...")
with zipfile.ZipFile(local_zip_path, 'r') as zip_ref:
    zip_ref.extractall(DATASET_PATH)
print(f"Fichiers extraits dans le dossier '{DATASET_PATH}'.")

In [ ]:
# (Optionnal) Remove zip file
os.remove(local_zip_path)

## Investigate and process the data

Let's start by reading in the credit card fraud data set.

In [ ]:
data = pd.read_csv(f"{DATASET_PATH}/creditcard.csv", delimiter=',')
data.head()

Let's take a peek at our data (we only show a subset of the columns in the table):

In [ ]:
print(data.columns)
data[['Time', 'V1', 'V2', 'V27', 'V28', 'Amount', 'Class']].describe()

This dataset consists entirely of numerical features since the original data was processed through PCA transformation to safeguard user privacy. The result is 28 PCA components labeled V1-V28, plus two unchanged features: _Amount_ (transaction value) and _Time_ (seconds between each transaction and the very first one in the dataset).

The class column indicates whether a transaction is fraudulent. We can see the data is heavily skewed toward legitimate transactions, with just 492 fraudulent cases (0.173%) among the 284,807 total examples.

In [ ]:
nonfrauds, frauds = data.groupby('Class').size()
print('Number of frauds: ', frauds)
print('Number of non-frauds: ', nonfrauds)
print('Percentage of fradulent data:', 100.*frauds/(frauds + nonfrauds))

We already know that the columns $V_i$ have been normalized to have $0$ mean and unit standard deviation as the result of a PCA.

In [ ]:
feature_columns = data.columns[:-1]
label_column = data.columns[-1]

features = data[feature_columns].values.astype('float32')
labels = (data[label_column].values).astype('float32')

Next, we will prepare our data for loading and training.

## Training

We will split our dataset into a train and test to evaluate the performance of our models. It's important to do so _before_ any techniques meant to alleviate the class imbalance are used. This ensures that we don't leak information from the test set into the train set.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.1, random_state=42
)

> Note: If you are bringing your own data to this solution and they include categorical data, that have strings as values, you'd need to one-hot encode these values first using for example sklearn's [OneHotEncoder](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-categorical-features), as XGBoost only supports numerical data.

## Unsupervised Learning

In [ ]:
import os
import sagemaker


# sagemaker_iam_role = os.getenv("SAGEMAKER_IAM_ROLE")
sagemaker_session = sagemaker.Session()
sagemaker_iam_role = sagemaker.get_execution_role()
default_bucket = sagemaker_session.default_bucket()

data_location = 's3://{}/{}/'.format(default_bucket, s3_prefix)
base_job_name = "{}-rcf".format(s3_prefix)
output_path = 's3://{}/{}/output'.format(default_bucket, s3_prefix)


print(sagemaker_iam_role)
print(default_bucket)
print('Training artifacts will be uploaded to: {}'.format(output_path))
data_location

Fraud detection typically comes with sparse labeled data, and obtaining fraud labels can be incredibly time-consuming. This is where we want to extract insights from our unlabeled data as well. _Anomaly detection_ uses unsupervised learning to identify outliers based entirely on feature patterns. Random Cut Forest represents a modern anomaly detection method that delivers both accuracy and scalability. We'll build one using our training data and test its effectiveness on our holdout set.

In [ ]:
from sagemaker import RandomCutForest

# specify general training job information
rcf = RandomCutForest(
    sagemaker_session=sagemaker_session,
    role=sagemaker_iam_role,
    instance_count=1,
    instance_type='ml.m4.xlarge',
    data_location=data_location,
    output_path=output_path,
    base_job_name=base_job_name,
    num_samples_per_tree=512,
    num_trees=50
)

Now we are ready to fit the model. The below cell should take around 5 minutes to complete.

In [ ]:
rcf.fit(rcf.record_set(X_train))

### Host Random Cut Forest

Now that our model is trained, let's deploy it and run some predictions on our test data. SageMaker will set up an instance and take care of the deployment process - expect around 10 minutes total. You'll watch the progress through `-` markers and see an exclamation point when everything's ready.

In [ ]:
rcf_predictor = rcf.deploy(
    model_name="{}-rcf".format(s3_prefix),
    endpoint_name="{}-rcf-endpoint".format(s3_prefix),
    initial_instance_count=1,
    instance_type='ml.m4.xlarge'
)

In [ ]:
from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

rcf_predictor.content_type = 'text/csv'
rcf_predictor.serializer = CSVSerializer()
rcf_predictor.accept = 'application/json'
rcf_predictor.deserializer = JSONDeserializer()

### Test Random Cut Forest

With the model deployed, let's see how it performs in terms of separating fraudulent from legitimate transactions.

In [ ]:
def predict_rcf(current_predictor, data, rows=500):
    split_array = np.array_split(data, int(data.shape[0] / float(rows) + 1))
    predictions = []
    for array in split_array:
        array_preds = [s['score'] for s in current_predictor.predict(array)['scores']]
        predictions.append(array_preds)

    return np.concatenate([np.array(batch) for batch in predictions])

In [ ]:
positives = X_test[y_test == 1]
positives_scores = predict_rcf(rcf_predictor, positives)

negatives = X_test[y_test == 0]
negatives_scores = predict_rcf(rcf_predictor, negatives)

In [ ]:
positives_scores

In [ ]:
negatives_scores

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
sns.set(color_codes=True)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, ax = plt.subplots(figsize=(8, 6))
sns.histplot(positives_scores, label='fraud', bins=20, ax=ax)
sns.histplot(negatives_scores, label='not-fraud', bins=20, ax=ax)
ax.legend()

The unsupervised model is already doing a decent job separating the classes - higher anomaly scores tend to line up with fraudulent transactions.

## Clean up

We will leave the unsupervised and base XGBoost endpoints running at the end of this notebook so we can handle incoming event streams using the Lambda function. The solution will automatically clean up the endpoints when deleted, however, don't forget to ensure the prediction endpoints are deleted when you're done. You can do that at the Amazon SageMaker console in the Endpoints page. Or you can run `predictor_name.delete_endpoint()` here.

In [ ]:
# Uncomment to clean up endpoints
rcf_predictor.delete_model()
rcf_predictor.delete_endpoint()
sm_client = boto3.client('sagemaker', region_name=aws_region)
waiter = sm_client.get_waiter('endpoint_deleted')
waiter.wait(EndpointName="{}-rcf-endpoint".format(s3_prefix))
